# EDA evidence for the report

Every number in the report's EDA sections is computed here. The rows are the backtest sets, 2024 and 2025 publications (January-September) pooled: 64,120 CVEs, 85 later exploited. The label is "added to KEV after the publish month and by December 31". Rates are given **per 1,000 CVEs** with exact 95% intervals, because positives are rare.

Columns marked *diagnostic* (`nvd_status`, `cvss3_source`) are used only to explain the data, never as model inputs.

In [1]:
import sys, warnings; warnings.filterwarnings("ignore")
from pathlib import Path
ROOT = next(p for p in [Path(".."), Path(".")] if (p / "ml" / "config.py").exists()).resolve()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
from scipy import stats
from statsmodels.stats.contingency_tables import StratifiedTable
from sklearn.metrics import roc_auc_score
from ml import data
from ml.featureset import FeatureBuilder
pd.set_option("display.width", 220)

con = data.connect()
tr_rows, va_rows = data.load_splits(con)
fb = FeatureBuilder(con)
L = pd.concat([fb.transform(tr_rows).assign(year=2024), fb.transform(va_rows).assign(year=2025)], ignore_index=True)
diag = con.sql("SELECT cve_id, nvd_status, cvss3_source FROM vulnerabilities").df()
L = L.merge(diag, on="cve_id", how="left")
print(f"{len(L):,} CVEs, {int(L.y.sum())} exploited ({L.y.mean() * 1000:.2f} per 1,000)")

def rate(d, label=None):
    n, x = len(d), int(d.y.sum())
    lo = stats.beta.ppf(0.025, x, n - x + 1) if x else 0.0
    hi = stats.beta.ppf(0.975, x + 1, n - x)
    return {"group": label, "CVEs": n, "exploited": x, "per 1,000": round(1000 * x / n, 2) if n else np.nan,
            "95% CI": f"{1000 * lo:.2f}-{1000 * hi:.2f}"}
def table(groups):
    return pd.DataFrame([rate(d, g) for g, d in groups]).set_index("group")

64,120 CVEs, 85 exploited (1.33 per 1,000)


## 1. Relationships: how the main signals combine

In [2]:
L["epss_band"] = pd.cut(L.epss_pct, [-0.01, 0.5, 0.9, 1.0], labels=["bottom half", "50-90th pct", "top 10%"])
print("Exploit code x EPSS band (EPSS percentile within the snapshot month):")
display(table([(f"exploit code: {'yes' if e else 'no'}, EPSS {b}", d) for (e, b), d in L.groupby(["any_exploit", "epss_band"], observed=True)]))
L["assigner_band"] = pd.qcut(L.log_assigner_kev_rate.rank(method="first"), 3, labels=["low", "middle", "high"])
print("Exploit code x issuer's past KEV rate (thirds):")
table([(f"exploit code: {'yes' if e else 'no'}, issuer history {b}", d) for (e, b), d in L.groupby(["any_exploit", "assigner_band"], observed=True)])

Exploit code x EPSS band (EPSS percentile within the snapshot month):


,CVEs,exploited,"per 1,000",95% CI
group,,,,
"exploit code: no, EPSS bottom half",35080,28,0.80,0.53-1.15
"exploit code: no, EPSS 50-90th pct",22327,12,0.54,0.28-0.94
"exploit code: no, EPSS top 10%",5319,11,2.07,1.03-3.70
"exploit code: yes, EPSS bottom half",548,8,14.60,6.32-28.56
"exploit code: yes, EPSS 50-90th pct",403,4,9.93,2.71-25.22
"exploit code: yes, EPSS top 10%",443,22,49.66,31.38-74.23


Exploit code x issuer's past KEV rate (thirds):


,CVEs,exploited,"per 1,000",95% CI
group,,,,
"exploit code: no, issuer history low",21108,1,0.05,0.00-0.26
"exploit code: no, issuer history middle",21175,9,0.43,0.19-0.81
"exploit code: no, issuer history high",20443,41,2.01,1.44-2.72
"exploit code: yes, issuer history low",265,0,0.00,0.00-13.82
"exploit code: yes, issuer history middle",198,5,25.25,8.25-57.94
"exploit code: yes, issuer history high",931,29,31.15,20.96-44.43


In [3]:
feats = ["any_exploit", "poc_github", "epss_pct", "log_assigner_kev_rate", "cvss_filled", "kw_unauth", "net_preauth", "log_batch_size"]
corr = L[feats].corr(method="spearman").round(2)
print("Spearman correlations (pairs with |rho| >= 0.3):")
print(pd.DataFrame([(a, b, corr.loc[a, b]) for i, a in enumerate(feats) for b in feats[i + 1:] if abs(corr.loc[a, b]) >= 0.3],
                   columns=["A", "B", "rho"]).to_string(index=False))
print("\nShare of CVEs with exploit code, by EPSS band:", L.groupby("epss_band", observed=True).any_exploit.mean().round(3).to_dict())

Spearman correlations (pairs with |rho| >= 0.3):
          A           B  rho
any_exploit  poc_github 0.93
cvss_filled net_preauth 0.39

Share of CVEs with exploit code, by EPSS band: {'bottom half': 0.015, '50-90th pct': 0.018, 'top 10%': 0.077}


## 2. Subgroups that behave differently

In [4]:
THIRD = {"cve@mitre.org", "disclosure@vulncheck.com", "security-advisories@github.com", "security@wordfence.com", "audit@patchstack.com", "cna@vuldb.com"}
L["old_id"] = (pd.to_datetime(L.published).dt.year - L.cve_id.str[4:8].astype(int)) >= 2
L["reviewed"] = L.nvd_status.isin(["Analyzed", "Modified"])
groups = [("all CVEs", L),
          ("WordPress in description", L[L.kw_wordpress == 1]), ("cross-site scripting (XSS)", L[L.kw_xss == 1]),
          ("old ID (reserved 2+ years before publication)", L[L.old_id]),
          ("not reviewed by NIST (diagnostic)", L[~L.reviewed]),
          ("third-party issuer (MITRE, VulnCheck, GitHub...)", L[L.assigner.isin(THIRD)]),
          ("vendor or other issuer", L[~L.assigner.isin(THIRD)]),
          ("Microsoft (secure@microsoft.com)", L[L.assigner == "secure@microsoft.com"]),
          ("published Jan-Feb 2025 (EPSS data gap)", L[(L.year == 2025) & (L.snap <= "2025-02")]),
          ("'remote code execution' in description", L[L.kw_rce == 1]),
          ("edge device words (VPN, firewall, gateway, router...)", L[L.kw_edge_device == 1])]
table(groups)

,CVEs,exploited,"per 1,000",95% CI
group,,,,
all CVEs,64120,85,1.33,1.06-1.64
WordPress in description,6232,0,0.00,0.00-0.59
cross-site scripting (XSS),10450,2,0.19,0.02-0.69
old ID (reserved 2+ years before publication),4263,2,0.47,0.06-1.69
not reviewed by NIST (diagnostic),16722,0,0.00,0.00-0.22
"third-party issuer (MITRE, VulnCheck, GitHub...)",32127,26,0.81,0.53-1.19
vendor or other issuer,31993,59,1.84,1.40-2.38
Microsoft (secure@microsoft.com),1683,7,4.16,1.67-8.55
published Jan-Feb 2025 (EPSS data gap),7931,13,1.64,0.87-2.80


## 3. Confounding checks

In [5]:
# 3a. CVSS presence and source vs. NIST review
sc = L[L.cvss3_score.notna()].copy()
sc["scored_by"] = np.where(sc.cvss3_source == "nvd@nist.gov", "NIST", "vendor / issuer")
print("Exploitation rate by who scored CVSS (diagnostic):")
display(table([(f"CVSS scored by {g}", d) for g, d in sc.groupby("scored_by")] + [("CVSS blank", L[L.cvss3_score.isna()])]))
print("Share of NIST-reviewed CVEs:", {g: f"{d.reviewed.mean():.0%}" for g, d in [("NIST-scored", sc[sc.scored_by == 'NIST']), ("vendor-scored", sc[sc.scored_by != 'NIST'])]})

Exploitation rate by who scored CVSS (diagnostic):


,CVEs,exploited,"per 1,000",95% CI
group,,,,
CVSS scored by NIST,30934,55,1.78,1.34-2.31
CVSS scored by vendor / issuer,31741,30,0.95,0.64-1.35
CVSS blank,1445,0,0.00,0.00-2.55


Share of NIST-reviewed CVEs: {'NIST-scored': '100%', 'vendor-scored': '52%'}


In [6]:
# 3b. Do keyword effects survive comparing CVEs from the SAME issuer? (Mantel-Haenszel odds ratio)
def crude_vs_within(col):
    a = L[L[col] == 1]; b = L[L[col] == 0]
    crude = (a.y.mean()) / (b.y.mean())
    tabs = []
    for _, g in L.groupby("assigner"):
        t = np.array([[((g[col] == 1) & (g.y == 1)).sum(), ((g[col] == 1) & (g.y == 0)).sum()],
                      [((g[col] == 0) & (g.y == 1)).sum(), ((g[col] == 0) & (g.y == 0)).sum()]])
        if t[:, 0].sum() > 0 and t[0].sum() > 0 and t[1].sum() > 0:
            tabs.append(t + 0.5)
    st = StratifiedTable(tabs)
    lo, hi = st.oddsratio_pooled_confint()
    return {"feature": col, "crude rate ratio": round(crude, 2), "within-issuer odds ratio (MH)": round(st.oddsratio_pooled, 2),
            "95% CI": f"{lo:.2f}-{hi:.2f}", "issuers with a positive": len(tabs)}
pd.DataFrame([crude_vs_within(c) for c in ["kw_unauth", "kw_rce", "kw_edge_device", "net_preauth", "cwe_path", "any_exploit"]]).set_index("feature")

,crude rate ratio,within-issuer odds ratio (MH),95% CI,issuers with a positive
feature,,,,
kw_unauth,7.00,7.15,4.17-12.24,26
kw_rce,3.88,1.95,1.29-2.95,30
kw_edge_device,2.77,1.77,0.96-3.26,20
net_preauth,4.41,4.42,2.97-6.57,31
cwe_path,4.88,5.04,3.06-8.30,25
any_exploit,30.00,21.47,13.83-33.34,26


In [7]:
# 3c. Time-to-deadline: the same model, less time to be added
L["months_left"] = (pd.to_datetime(L.year.astype(str) + "-12-31") - (pd.to_datetime(L.snap) + pd.offsets.MonthEnd(0))).dt.days / 30.44
L["window"] = pd.cut(L.months_left, [0, 4, 7, 12], labels=["3-4 months left (Jul-Sep)", "5-7 months left (Apr-Jun)", "8-11 months left (Jan-Mar)"])
display(table([(str(g), d) for g, d in L.groupby("window", observed=True)]))
kev = L[L.y == 1].copy()
kev["delay_days"] = (pd.to_datetime(kev.kev_date) - (pd.to_datetime(kev.snap) + pd.offsets.MonthEnd(0))).dt.days
print("Days from the end of the publish month to KEV (exploited CVEs): median", int(kev.delay_days.median()),
      "| within 60 days:", f"{(kev.delay_days <= 60).mean():.0%}", "| within 120 days:", f"{(kev.delay_days <= 120).mean():.0%}")

,CVEs,exploited,"per 1,000",95% CI
group,,,,
3-4 months left (Jul-Sep),6821,11,1.61,0.81-2.88
5-7 months left (Apr-Jun),20128,29,1.44,0.97-2.07
8-11 months left (Jan-Mar),37171,45,1.21,0.88-1.62


Days from the end of the publish month to KEV (exploited CVEs): median 38 | within 60 days: 66% | within 120 days: 82%


In [8]:
# 3d. EPSS model change (March 2025)
for name, d in [("2024", L[L.year == 2024]), ("2025 Jan-Feb", L[(L.year == 2025) & (L.snap <= "2025-02")]), ("2025 Mar-Sep", L[(L.year == 2025) & (L.snap >= "2025-03")])]:
    print(f"{name:13} EPSS-percentile AUC {roc_auc_score(d.y, d.epss_pct):.2f} | raw EPSS median {d.epss.median():.5f} | "
          f"EPSS missing {d.epss.isna().mean():.0%} | {int(d.y.sum())} exploited")

2024          EPSS-percentile AUC 0.50 | raw EPSS median 0.00044 | EPSS missing 4% | 33 exploited
2025 Jan-Feb  EPSS-percentile AUC 0.57 | raw EPSS median 0.00043 | EPSS missing 35% | 13 exploited
2025 Mar-Sep  EPSS-percentile AUC 0.74 | raw EPSS median 0.00034 | EPSS missing 3% | 39 exploited


## 4. Non-linear relationships

In [9]:
def binned(col, bins, labels):
    b = pd.cut(L[col], bins, labels=labels, include_lowest=True)
    return table([(str(g), d) for g, d in L.groupby(b, observed=True)])
print("EPSS percentile within the month (deciles):")
L["epss_dec"] = pd.qcut(L.epss_pct.rank(method="first"), 10, labels=[f"D{i}" for i in range(1, 11)])
display(table([(str(g), d) for g, d in L.groupby("epss_dec", observed=True)]))
print("Issuer's past KEV rate (deciles):")
L["asg_dec"] = pd.qcut(L.log_assigner_kev_rate.rank(method="first"), 10, labels=[f"D{i}" for i in range(1, 11)])
display(table([(str(g), d) for g, d in L.groupby("asg_dec", observed=True)]))

EPSS percentile within the month (deciles):


,CVEs,exploited,"per 1,000",95% CI
group,,,,
D1,6412,11,1.72,0.86-3.07
D2,6412,4,0.62,0.17-1.60
D3,6412,9,1.40,0.64-2.66
D4,6412,6,0.94,0.34-2.04
D5,6412,5,0.78,0.25-1.82
D6,6412,4,0.62,0.17-1.60
D7,6412,1,0.16,0.00-0.87
D8,6412,7,1.09,0.44-2.25
D9,6412,4,0.62,0.17-1.60


Issuer's past KEV rate (deciles):


,CVEs,exploited,"per 1,000",95% CI
group,,,,
D1,6412,0,0.00,0.00-0.58
D2,6412,0,0.00,0.00-0.58
D3,6412,1,0.16,0.00-0.87
D4,6412,1,0.16,0.00-0.87
D5,6412,0,0.00,0.00-0.58
D6,6412,10,1.56,0.75-2.87
D7,6412,8,1.25,0.54-2.46
D8,6412,6,0.94,0.34-2.04
D9,6412,20,3.12,1.91-4.81


In [10]:
print("Release batch size:")
display(binned("batch_size", [0, 1, 3, 10, 50, 1e9], ["1 (lone)", "2-3", "4-10", "11-50", "51+"]))
print("CVSS score (blank shown separately):")
display(table([(str(g), d) for g, d in L[L.cvss3_score.notna()].groupby(pd.cut(L.cvss3_score, [0, 6.9, 8.9, 9.7, 9.8, 10], labels=["< 7", "7-8.9", "9-9.7", "9.8", "9.9-10"]), observed=True)]))
print(f"Share of 9+ scores that are exactly 9.8: {(L.cvss3_score == 9.8).sum() / (L.cvss3_score >= 9).sum():.0%}")

Release batch size:


,CVEs,exploited,"per 1,000",95% CI
group,,,,
1 (lone),2692,17,6.32,3.68-10.09
2-3,4087,26,6.36,4.16-9.31
4-10,10567,16,1.51,0.87-2.46
11-50,26624,19,0.71,0.43-1.11
51+,20150,7,0.35,0.14-0.72


CVSS score (blank shown separately):


,CVEs,exploited,"per 1,000",95% CI
group,,,,
< 7,32052,8,0.25,0.11-0.49
7-8.9,22353,38,1.70,1.20-2.33
9-9.7,1170,6,5.13,1.88-11.13
9.8,6766,28,4.14,2.75-5.98
9.9-10,333,5,15.02,4.89-34.69


Share of 9+ scores that are exactly 9.8: 82%
